## baseline physics model


In [ ]:
from utils import calculate_nse, create_lag

In [ ]:
import pandas as pd
df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
pd.to_datetime(df['date'])
df = df.set_index('date')
df.drop(columns= 'Unnamed: 0', inplace= True)
df.drop(columns= 'Unnamed: 0.1', inplace= True)

df.head()

In [ ]:
df1 = df.copy()

In [ ]:
#for single column dataframes always use double brackets, single brackets will make it a series, or float type
# for adding two columns to a df either add them one by one or use two brackets

### reduce to a single function to test multiple combinations of lag

In [1]:
# for lag in range(1,8): 1 2 6 3
from sklearn.preprocessing import StandardScaler

lag_scaler = StandardScaler()


x_lagged = create_lag(df1, lag_precip=1, lag_dd=2, lag_sca=2, lag_et=1)

x_train = x_lagged[:'2017-12-31']
x_val = x_lagged['2018-01-01':'2021-12-31']
x_test = x_lagged['2022-01-01':]

y_train = x_lagged[:'2017-12-31']
y_val = x_lagged['2018-01-01':'2021-12-31']
y_test = x_lagged['2022-01-01':]

X_train_lagged = x_train[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_train_lagged = y_train[['runoff']]
Y_val_lagged = y_val[['runoff']]
X_val_lagged = x_val[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
X_test_lagged = x_test[['precipitation','precipitation_lagged_cum','melt_proxy','et_loss_lagged']]
Y_test_lagged = x_test[['runoff']]

lag_scaler.fit(X_train_lagged)
 
x_train_scaled = lag_scaler.transform(X_train_lagged)
x_val_scaled = lag_scaler.transform(X_val_lagged)
x_test_scaled = lag_scaler.transform(X_test_lagged)

amodel = LinearRegression()
amodel.fit(X_train_lagged, Y_train_lagged)
# amodel.fit(X_val_lagged, Y_val_lagged)

y_val_pred = amodel.predict(X_val_lagged)
baseline_nse = calculate_nse(Y_val_lagged, y_val_pred)
print(baseline_nse)
# alpha, beta, gamma, delta=amodel.coef_
print(amodel.coef_)

NameError: name 'create_lag' is not defined

In [ ]:
#accumulated precipitation
#5 month - 11->3, 3 month window - 12-2
# df1['precipitation_acc'] = df1['precipitation']
# for precip in range(2000,2026):
#     df1['precipitation_acc'] = df1['precipitation'] 
df1.index = pd.to_datetime(df1.index)

winter_months = df1[df1.index.month.isin([11, 12, 1, 2, 3])]
print(winter_months['precipitation'].head())
print(f"Total winter records: {len(winter_months)}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from itertools import product

def create_lag_varied(df, lag_precip, lag_sca, lag_dd, lag_et):
    """Create lagged features with different lags for each variable"""
    df_copy = df.copy()
    
    df_copy['precipitation_lagged'] = df_copy['precipitation'].shift(lag_precip)
    df_copy['sca_lagged'] = df_copy['sca'].shift(lag_sca)
    df_copy['dd_lagged'] = df_copy['dd'].shift(lag_dd)
    df_copy['et_loss_lagged'] = df_copy['et_loss'].shift(lag_et)
    df_copy['melt_proxy'] = df_copy['sca_lagged'] * df_copy['dd_lagged']
    
    df_clean = df_copy.dropna()
    return df_clean

def calculate_nse(observed, predicted):
    mean_obs = np.mean(observed)
    numerator = np.sum((observed - predicted) ** 2)
    denominator = np.sum((observed - mean_obs) ** 2)
    return 1 - (numerator / denominator)

# Load data
df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
df = df.set_index('date')
df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'], inplace=True, errors='ignore')

# Test all combinations
results = []

# Generate all combinations: 0-7 for each feature = 8^4 = 4096 combinations
# lag_range = range(1, 8)
for lag_precip, lag_sca, lag_dd, lag_et in product(range(0,1),
                                                   range(0,3),
                                                   range(0,2),
                                                   range(0,2)

    ):
    try:
        # Create lagged features
        x = create_lag_varied(df, lag_precip, lag_sca, lag_dd, lag_et)
        
        # Select features and target
        X = x[['precipitation','precipitation_lagged', 'melt_proxy', 'et_loss_lagged']]
        y = x['runoff']
        
        # Split data
        X_train = X[:'2017-12-31']
        X_val = X['2018-01-01':'2021-12-31']
        
        y_train = y[:'2017-12-31']
        y_val = y['2018-01-01':'2021-12-31']
        
        # Skip if not enough data
        if len(X_train) < 10 or len(X_val) < 10:
            continue
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)
        
        # Train model
        model = LinearRegression()
        model.fit(X_train_scaled, y_train)
        
        # Predict and evaluate
        y_val_pred = model.predict(X_val_scaled)
        nse = calculate_nse(y_val, y_val_pred)
        
        # Store results
        results.append({
            'lag_precip': lag_precip,
            'lag_sca': lag_sca,
            'lag_dd': lag_dd,
            'lag_et': lag_et,
            'nse': nse
        })
    except Exception as e:
        continue

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df.head()
# Find best combination
best_result = results_df.loc[results_df['nse'].idxmax()]

print(f"\nBest NSE: {best_result['nse']:.4f}")
print(f"Best lag combination:")
print(f"  Precipitation lag: {int(best_result['lag_precip'])}")
print(f"  SCA lag: {int(best_result['lag_sca'])}")
print(f"  DD lag: {int(best_result['lag_dd'])}")
print(f"  ET loss lag: {int(best_result['lag_et'])}")

# Save all results
results_df.to_csv('lag_optimization_results.csv', index=False)

# Show top 10 combinations
print("\nTop 10 lag combinations:")
print(results_df.nlargest(50, 'nse'))

In [ ]:
x_lagged_x = create_lag(df1, lag_precip=1, lag_dd=3, lag_sca=3, lag_et=1)
print(x_lagged_x[['melt_proxy','sca_lagged','dd_lagged', 'runoff']].corr()['runoff'])


In [ ]:
# Run this and tell me the output
print("Data summary:")
print(x_lagged[['precipitation', 'sca', 'dd', 'et_loss','melt_proxy','sca_lagged','dd_lagged', 'runoff']].describe())

print("\nCorrelations with discharge:")
print(x_lagged[['precipitation', 'sca', 'dd', 'et_loss','melt_proxy','sca_lagged','dd_lagged', 'runoff']].corr()['runoff'])

print("\nFirst 5 rows:")
print(x_lagged[['precipitation', 'sca', 'dd', 'et_loss','melt_proxy','sca_lagged','dd_lagged', 'runoff']].head())

In [ ]:
corr_matrix = x.corr()
corr_matrix

In [ ]:
x_lagged

In [ ]:
corr = x_lagged[['precipitation', 'sca', 'dd', 'et_loss','melt_proxy','sca_lagged','dd_lagged', 'runoff']].corr()['runoff']
print(corr)

In [ ]:
print("SCA seasonal pattern:")
x_lagged.set_index('date')
print(x_lagged.groupby(x_lagged.index.month)['SCA'].mean())
   # If SCA is HIGH in winter, LOW in summer → it's snow cover %
   # This is your problem!

In [ ]:
# Plot heatmap
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            vmin=-1, vmax=1, square=True, linewidths=1)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import LinearRegression
# from sklearn.preprocessing import StandardScaler
# from itertools import product
# # MAX_LAG = 7
# # df = df.iloc[MAX_LAG:]


# def create_lag_varied(df, lag_precip, lag_sca, lag_dd, lag_et):
#     """Create lagged features with different lags for each variable"""
#     df_copy = df.copy()
    
#     df_copy['precipitation_lagged'] = df_copy['precipitation'].shift(lag_precip)
#     df_copy['sca_lagged'] = df_copy['sca'].shift(lag_sca)
#     df_copy['dd_lagged'] = df_copy['dd'].shift(lag_dd)
#     df_copy['et_loss_lagged'] = df_copy['et_loss'].shift(lag_et)
#     df_copy['melt_proxy'] = df_copy['sca_lagged'] * df_copy['dd_lagged']
    
#     df_clean = df_copy.dropna()
#     return df_clean

# def calculate_nse(observed, predicted):
#     mean_obs = np.mean(observed)
#     numerator = np.sum((observed - predicted) ** 2)
#     denominator = np.sum((observed - mean_obs) ** 2)
#     return 1 - (numerator / denominator)

# # Load data
# df = pd.read_csv('final_data_physics_baseline/phy_final_corrected_data.csv')
# df = df.set_index('date')
# df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'], inplace=True, errors='ignore')

# # Test all combinations
# results = []

# # Generate all combinations: 0-7 for each feature = 8^4 = 4096 combinations
# lag_range = range(1, 8)

# for lag_precip, lag_sca, lag_dd, lag_et in product(lag_range, repeat=4):
#     try:
#         # Create lagged features
#         x = create_lag_varied(df, lag_precip, lag_sca, lag_dd, lag_et)
        
#         # Select features and target
#         X = x[['precipitation_lagged', 'melt_proxy', 'et_loss_lagged']]
#         y = x['runoff']
        
#         # Split data
#         X_train = X[:'2017-12-31']
#         X_val = X['2018-01-01':'2021-12-31']
        
#         y_train = y[:'2017-12-31']
#         y_val = y['2018-01-01':'2021-12-31']
        
#         # Skip if not enough data
#         if len(X_train) < 10 or len(X_val) < 10:
#             continue
        
#         # Scale features
#         scaler = StandardScaler()
#         X_train_scaled = scaler.fit_transform(X_train)
#         X_val_scaled = scaler.transform(X_val)
        
#         # Train model
#         model = LinearRegression()
#         model.fit(X_train_scaled, y_train)
        
#         # Predict and evaluate
#         y_val_pred = model.predict(X_val_scaled)
#         nse = calculate_nse(y_val, y_val_pred)
        
#         # Store results
#         results.append({
#             'lag_precip': lag_precip,
#             'lag_sca': lag_sca,
#             'lag_dd': lag_dd,
#             'lag_et': lag_et,
#             'nse': nse
#         })
        
#     except Exception as e:
#         continue

# # Convert to DataFrame
# results_df = pd.DataFrame(results)
# results_df.head()
# # Find best combination
# best_result = results_df.loc[results_df['nse'].idxmax()]

# print(f"\nBest NSE: {best_result['nse']:.4f}")
# print(f"Best lag combination:")
# print(f"  Precipitation lag: {int(best_result['lag_precip'])}")
# print(f"  SCA lag: {int(best_result['lag_sca'])}")
# print(f"  DD lag: {int(best_result['lag_dd'])}")
# print(f"  ET loss lag: {int(best_result['lag_et'])}")

# # Save all results
# results_df.to_csv('lag_optimization_results.csv', index=False)

# # Show top 10 combinations
# print("\nTop 10 lag combinations:")
# print(results_df.nlargest(50, 'nse'))

In [ ]:
x_lagged_x = create_lag(df1, lag_precip=1, lag_dd=4, lag_sca=2, lag_et=1)
print(x_lagged_x[['melt_proxy','sca_lagged','dd_lagged', 'runoff']].corr()['runoff'])
